# Week 3 — Data contract, five features, and the leakage trap

**Lane:** Refresh / Content Opportunity Scoring

This notebook defines the lane's warehouse slice, verifies three facts with three small March 2026 queries, builds exactly five decision-time features, and demonstrates why a label-derived feature must be removed.

## 1–2. My contract in five plain answers

1. **What one row means:** one pseudonymized content item at the end of March 2026, ready to be ranked for human refresh review.
2. **Tables used:** `fact_content_daily_performance` supplies daily Search Console measurements; `dim_content` supplies the safe creation date. I aggregate daily facts to one content-month row.
3. **Time window:** March 1–31, 2026 supplies features. April 1–30, 2026 supplies the later outcome used only as a label. June 2026 remains sealed and is not used here.
4. **What I would rank:** pages by the risk of a meaningful next-month visibility decline. The proxy label is `future_decline = 1` when April impressions are below 80% of March impressions, with at least 20 measured GSC days in both months.
5. **What I deliberately exclude:** every April metric from the predictive feature set. It is future information at the March decision moment and would leak the answer. IDs are retained only as context for joins and grouped splitting.

**Output and action:** a ranked, human-review queue. An editor could inspect high-risk pages and decide whether to protect, refresh, expand, consolidate, or monitor them; the score never triggers an automatic content change.

In [1]:
from pathlib import Path
import os

import duckdb
import numpy as np
import pandas as pd
from huggingface_hub import get_token
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
extension_dir = repo_root / "work" / "outputs" / ".duckdb_extensions"
extension_dir.mkdir(parents=True, exist_ok=True)

hf_token = os.environ.get("HF_TOKEN") or get_token()
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
assert hf_token, "Store a Hugging Face read token as the HF_TOKEN secret; never paste it into a cell."

con = duckdb.connect()
con.execute(f"SET extension_directory='{extension_dir.as_posix()}'")
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [hf_token])

ROOT = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH = f"read_parquet('{ROOT}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_APRIL = f"read_parquet('{ROOT}/fact_content_daily_performance/month=2026-04/*.parquet')"
DIM_CONTENT = f"read_parquet('{ROOT}/dim_content.parquet')"

print("Authenticated without displaying the token.")
print("Development month: March 2026 | outcome month: April 2026 | June remains sealed")

Authenticated without displaying the token.
Development month: March 2026 | outcome month: April 2026 | June remains sealed


## 3. Three verification queries

These are the **exactly three** small verification queries requested. They operate on the March 2026 mid-panel partition, not the final-month `_sample` table.

### Verification query 1 — grain

The published daily grain is `report_date × client_hash_id × content_hash_id`. If the query below returns zero rows, that grain is unique in the March slice.

In [2]:
grain_query = f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS duplicate_count
FROM {FACT_MARCH}
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5
"""
grain_violations = con.sql(grain_query).df()
display(grain_violations)
print(f"Duplicate grain groups returned: {len(grain_violations)}")
assert grain_violations.empty

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_count


Duplicate grain groups returned: 0


### Verification query 2 — slice size and date span

This proves how many daily rows, clients, and content items are in the March partition and confirms that the partition spans March 1–31.

In [3]:
slice_query = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items
FROM {FACT_MARCH}
"""
slice_facts = con.sql(slice_query).df()
display(slice_facts)

,row_count,min_date,max_date,clients,content_items
0,9841378,2026-03-01,2026-03-31,55,331437


### Verification query 3 — availability

The `available` CTE uses the required `IS TRUE` filter. Rows that fail this measurement flag are not interpreted as zero search activity.

In [4]:
availability_query = f"""
WITH all_rows AS (
    SELECT COUNT(*) AS total_rows FROM {FACT_MARCH}
), available AS (
    SELECT * FROM {FACT_MARCH}
    WHERE gsc_data_available IS TRUE
)
SELECT
    all_rows.total_rows,
    COUNT(*) AS rows_surviving_is_true,
    ROUND(100.0 * COUNT(*) / all_rows.total_rows, 2) AS percent_surviving
FROM available
CROSS JOIN all_rows
GROUP BY all_rows.total_rows
"""
availability_facts = con.sql(availability_query).df()
display(availability_facts)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_surviving_is_true,percent_surviving
0,9841378,3611061,36.69


## Five-feature frame

I keep **exactly five** predictive features:

1. **`impressions`** — knowable at the decision moment because it sums measured GSC exposure during March only.
2. **`ctr`** — knowable at the decision moment because March clicks and impressions have already occurred; it is missing when the denominator is zero.
3. **`avg_position`** — knowable at the decision moment because it is the March impression-weighted position over valid measured rows only.
4. **`active_days`** — knowable at the decision moment because it counts March days with measured positive impressions.
5. **`content_age_days`** — knowable at the decision moment because creation date is metadata already known by March 31.

Client and content hashes are context, not model features. `future_decline` is the April label, not a feature.

In [5]:
feature_query = f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr,
        SUM(gsc_sum_position) FILTER (
            WHERE gsc_impressions > 0 AND gsc_sum_position > 0
        ) / NULLIF(SUM(gsc_impressions) FILTER (
            WHERE gsc_impressions > 0 AND gsc_sum_position > 0
        ), 0) AS avg_position,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS active_days,
        COUNT(*) AS available_days
    FROM {FACT_MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
), april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS outcome_impressions,
        COUNT(*) AS outcome_available_days
    FROM {FACT_APRIL}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
), content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        MIN(content_created_date) AS content_created_date
    FROM {DIM_CONTENT}
    GROUP BY 1, 2
)
SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.impressions,
    m.ctr,
    m.avg_position,
    m.active_days,
    GREATEST(DATE_DIFF('day', c.content_created_date, DATE '2026-03-31'), 0) AS content_age_days,
    CASE WHEN a.outcome_impressions < 0.80 * m.impressions THEN 1 ELSE 0 END AS future_decline
FROM march AS m
JOIN april AS a USING (client_hash_id, content_hash_id)
LEFT JOIN content AS c USING (client_hash_id, content_hash_id)
WHERE m.impressions >= 100
  AND m.available_days >= 20
  AND a.outcome_available_days >= 20
ORDER BY m.client_hash_id, m.content_hash_id
"""

examples = con.sql(feature_query).df()
feature_names = ["impressions", "ctr", "avg_position", "active_days", "content_age_days"]
feature_frame = examples[["client_hash_id", "content_hash_id", *feature_names, "future_decline"]].copy()

print(f"Eligible content rows: {len(feature_frame):,}")
print(f"Observed decline rate: {feature_frame['future_decline'].mean():.3f}")
print(f"Predictive feature count: {len(feature_names)}")
assert len(feature_names) == 5
display(feature_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible content rows: 88,941
Observed decline rate: 0.513
Predictive feature count: 5


,client_hash_id,content_hash_id,impressions,ctr,avg_position,active_days,content_age_days,future_decline
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,331.0,0.006042,14.377644,31,175,0
1,client_0797ff3a1fc9a6a5,content_1207efddce873942,461.0,0.000000,14.488069,31,175,0
2,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,232.0,0.000000,11.961207,29,175,0
3,client_0797ff3a1fc9a6a5,content_37952b007ab057b3,774.0,0.006460,12.746770,31,175,0
4,client_0797ff3a1fc9a6a5,content_7beb639d1052e49e,311.0,0.000000,8.540193,30,175,0


## The deliberate leakage trap

For the lesson, I add one forbidden column—an exact copy of the later label—then compare it with the honest five-feature model on the same client-grouped holdout. A near-perfect leaked score is not evidence of a useful model; it only shows that the answer was smuggled into the inputs. I then delete that one column and retain the honest score.

In [6]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(feature_frame, groups=feature_frame["client_hash_id"]))

y = feature_frame["future_decline"]

def quick_auc(columns):
    model = Pipeline([
        ("prep", ColumnTransformer([
            ("numeric", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scale", StandardScaler()),
            ]), columns)
        ])),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ])
    model.fit(feature_frame.iloc[train_idx][columns], y.iloc[train_idx])
    probabilities = model.predict_proba(feature_frame.iloc[test_idx][columns])[:, 1]
    return roc_auc_score(y.iloc[test_idx], probabilities)

honest_auc = quick_auc(feature_names)

leaked_frame = feature_frame.copy()
leaked_frame["future_decline_copy"] = leaked_frame["future_decline"]  # deliberately forbidden
feature_frame = leaked_frame
leaked_auc = quick_auc([*feature_names, "future_decline_copy"])

feature_frame = feature_frame.drop(columns="future_decline_copy")
assert "future_decline_copy" not in feature_frame.columns

score_comparison = pd.DataFrame({
    "version": ["honest five features (kept)", "label copy included (rejected)"],
    "grouped_holdout_roc_auc": [honest_auc, leaked_auc],
})
display(score_comparison.style.format({"grouped_holdout_roc_auc": "{:.3f}"}))
print(f"Honest AUC retained: {honest_auc:.3f}")
print(f"Leaked AUC rejected: {leaked_auc:.3f}")
print("Final predictive columns:", feature_names)
print("Leak column present after cleanup:", "future_decline_copy" in feature_frame.columns)

,version,grouped_holdout_roc_auc
0,honest five features (kept),0.664
1,label copy included (rejected),1.000


Honest AUC retained: 0.664
Leaked AUC rejected: 1.000
Final predictive columns: ['impressions', 'ctr', 'avg_position', 'active_days', 'content_age_days']
Leak column present after cleanup: False


## 4. One named limitation

**Coverage-selection limitation:** clients begin GSC tracking at different times, and only rows with `gsc_data_available IS TRUE` survive. The eligible March-to-April slice therefore represents pages with sufficient measured coverage, not every page or every client in the warehouse. The label records an observed association and cannot prove that a refresh would cause recovery or reveal Google's ranking algorithm.

## 5. Self-check

- [x] Five plain-language contract answers are present.
- [x] Exactly three verification queries are shown with visible outputs.
- [x] Availability is filtered with `IS TRUE` and the surviving count is displayed.
- [x] March 2026 is the mid-panel development month; June remains sealed.
- [x] The feature frame contains exactly five predictive features, each with an availability line.
- [x] One label-derived column is deliberately added, scored, deleted, and rejected.
- [x] The honest grouped-holdout score is retained.
- [x] One limitation is named, and the output supports a real human content action.
- [x] No token, client name, domain, URL, raw query, or private export is included.